# 카드 생성 테스트

`data/news/` → 뉴스 카드  
`data/policy/` → 정책 카드  
출력: `data/cards/news/`, `data/cards/policy/`

## 셀 1. 환경 설정

In [1]:
import sys, json, logging, os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# ── 경로 설정 ─────────────────────────────────────────────────────────────
AI_AGENT_DIR = Path(r"C:\Users\skybl\OneDrive\바탕 화면\FINAL\ai_agent")  # 본인 경로로 수정
sys.path.insert(0, str(AI_AGENT_DIR))

DATA_DIR      = AI_AGENT_DIR / "data"
NEWS_DATA_DIR = DATA_DIR / "news"
POL_DATA_DIR  = DATA_DIR / "policy"

# 출력 디렉토리
OUT_NEWS_DIR   = DATA_DIR / "cards" / "news"
OUT_POLICY_DIR = DATA_DIR / "cards" / "policy"
OUT_NEWS_DIR.mkdir(parents=True, exist_ok=True)
OUT_POLICY_DIR.mkdir(parents=True, exist_ok=True)

# ── 로깅 ──────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

print(f"AI_AGENT_DIR : {AI_AGENT_DIR.exists()} | {AI_AGENT_DIR}")
print(f"NEWS_DATA    : {NEWS_DATA_DIR.exists()} | {NEWS_DATA_DIR}")
print(f"POLICY_DATA  : {POL_DATA_DIR.exists()} | {POL_DATA_DIR}")
print(f"OUT_NEWS     : {OUT_NEWS_DIR}")
print(f"OUT_POLICY   : {OUT_POLICY_DIR}")

AI_AGENT_DIR : True | C:\Users\skybl\OneDrive\바탕 화면\FINAL\ai_agent
NEWS_DATA    : False | C:\Users\skybl\OneDrive\바탕 화면\FINAL\ai_agent\data\news
POLICY_DATA  : False | C:\Users\skybl\OneDrive\바탕 화면\FINAL\ai_agent\data\policy
OUT_NEWS     : C:\Users\skybl\OneDrive\바탕 화면\FINAL\ai_agent\data\cards\news
OUT_POLICY   : C:\Users\skybl\OneDrive\바탕 화면\FINAL\ai_agent\data\cards\policy


## 셀 2. 데이터 로드

In [3]:
# ── 뉴스 데이터 — 키워드별 JSON 파일 ─────────────────────────────────────
news_files = sorted(NEWS_DATA_DIR.glob("naver_*_clean.json"))
NEWS_BY_KEYWORD: dict[str, list] = {}

for fpath in news_files:
    with open(fpath, encoding="utf-8") as f:
        articles = json.load(f)
    keyword = articles[0]["keyword_matched"] if articles else fpath.stem
    NEWS_BY_KEYWORD[keyword] = articles
    print(f"  뉴스 로드: [{keyword}] {len(articles)}건")

print(f"\n뉴스 키워드 {len(NEWS_BY_KEYWORD)}개 로드 완료")

# ── 정책 데이터 ────────────────────────────────────────────────────────────
pol_file = sorted(POL_DATA_DIR.glob("gov24_top5_clean*.json"))[-1]
with open(pol_file, encoding="utf-8") as f:
    POLICY_DATA: dict = json.load(f)  # {카테고리: [정책, ...]}

total_policies = sum(len(v) for v in POLICY_DATA.values())
print(f"\n정책 데이터: {pol_file.name}")
for cat, pols in POLICY_DATA.items():
    print(f"  [{cat}] {len(pols)}건")

# ── 정책 관련 뉴스 (keyword_matched로 매핑) ───────────────────────────────
related_news_file = POL_DATA_DIR / "related_articles" / "cleaned_news.jsonl"
RELATED_NEWS: list[dict] = []
if related_news_file.exists():
    with open(related_news_file, encoding="utf-8") as f:
        RELATED_NEWS = [json.loads(l) for l in f if l.strip()]

# keyword_matched 기준으로 인덱싱
RELATED_BY_KEYWORD: dict[str, list] = {}
for art in RELATED_NEWS:
    kw = art.get("keyword_matched", "")
    RELATED_BY_KEYWORD.setdefault(kw, []).append(art)

print(f"\n관련 뉴스: {len(RELATED_NEWS)}건 ({len(RELATED_BY_KEYWORD)}개 키워드)")

law_file = POL_DATA_DIR / "related_bills" / "law_clean.json"
with open(law_file, encoding="utf-8") as f:
    LAW_DATA = json.load(f)
print(f"법령 데이터: {len(LAW_DATA)}건")

  뉴스 로드: [결혼세액공제] 526건
  뉴스 로드: [고유가 피해지원금] 660건
  뉴스 로드: [국민배당금] 356건
  뉴스 로드: [레버리지ETF] 756건
  뉴스 로드: [설탕 부담금] 534건

뉴스 키워드 5개 로드 완료

정책 데이터: gov24_top5_clean_2026-06-06.json
  [일자리] 5건
  [교육] 5건
  [주거] 5건
  [금융] 5건
  [생활복지] 5건
  [문화] 5건

관련 뉴스: 2363건 (29개 키워드)
법령 데이터: 22건


## 셀 3. 생성기 초기화

In [4]:
from openai import OpenAI
from db.qdrant_connect import get_qdrant_client
from agents.card_generation.news import NewsCardGenerator
from agents.card_generation.policy import PolicyCardGenerator
from config import OPENAI_API_KEY

QDRANT_PATH = str(AI_AGENT_DIR / "qdrant_storage")

openai_client = OpenAI(api_key=OPENAI_API_KEY)
qdrant_client = get_qdrant_client("local", local_path=QDRANT_PATH)

news_gen   = NewsCardGenerator(qdrant_client=qdrant_client, openai_client=openai_client)
policy_gen = PolicyCardGenerator(qdrant_client=qdrant_client, openai_client=openai_client)

print("✅ 생성기 초기화 완료")
print(f"  Qdrant: {QDRANT_PATH}")

[Qdrant] 로컬 모드: C:\Users\skybl\OneDrive\바탕 화면\FINAL\ai_agent\qdrant_storage


14:38:49 [INFO] HTTP Request: HEAD https://huggingface.co/BarryKim34/kr-electra-political-bias/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
14:38:49 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BarryKim34/kr-electra-political-bias/44b4e6c54e4ef3a7a3adfcaca4cbeaf23319dd85/config.json "HTTP/1.1 200 OK"
14:38:49 [INFO] HTTP Request: HEAD https://huggingface.co/BarryKim34/kr-electra-political-bias/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
14:38:49 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BarryKim34/kr-electra-political-bias/44b4e6c54e4ef3a7a3adfcaca4cbeaf23319dd85/tokenizer_config.json "HTTP/1.1 200 OK"
14:38:50 [INFO] HTTP Request: HEAD https://huggingface.co/BarryKim34/kr-electra-political-bias/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
14:38:50 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BarryKim34/kr-electra-political-bias/44b4e6

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

14:38:52 [INFO] HTTP Request: GET https://huggingface.co/api/models/BarryKim34/kr-electra-political-bias "HTTP/1.1 200 OK"
14:38:52 [INFO] HTTP Request: GET https://huggingface.co/api/models/BarryKim34/kr-electra-political-bias/commits/main "HTTP/1.1 200 OK"
14:38:52 [INFO] HTTP Request: GET https://huggingface.co/api/models/BarryKim34/kr-electra-political-bias/discussions?p=0 "HTTP/1.1 200 OK"
14:38:52 [INFO] BiasClassifier 로드 완료: BarryKim34/kr-electra-political-bias / cuda
14:38:52 [INFO] HTTP Request: GET https://huggingface.co/api/models/BarryKim34/kr-electra-political-bias/commits/refs%2Fpr%2F1 "HTTP/1.1 200 OK"
14:38:52 [INFO] HTTP Request: HEAD https://huggingface.co/BarryKim34/kr-electra-political-bias/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
14:38:52 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BarryKim34/kr-electra-political-bias/44b4e6c54e4ef3a7a3adfcaca4cbeaf23319dd85/config.json "HTTP/1.1 200 OK"
14:38:52 [INFO] HTTP Request: 

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

14:38:54 [INFO] HTTP Request: GET https://huggingface.co/api/models/BarryKim34/kr-electra-political-bias "HTTP/1.1 200 OK"
14:38:54 [INFO] BiasClassifier 로드 완료: BarryKim34/kr-electra-political-bias / cuda


✅ 생성기 초기화 완료
  Qdrant: C:\Users\skybl\OneDrive\바탕 화면\FINAL\ai_agent\qdrant_storage


14:38:54 [INFO] HTTP Request: GET https://huggingface.co/api/models/BarryKim34/kr-electra-political-bias/commits/main "HTTP/1.1 200 OK"
14:38:55 [INFO] HTTP Request: GET https://huggingface.co/api/models/BarryKim34/kr-electra-political-bias/discussions?p=0 "HTTP/1.1 200 OK"
14:38:55 [INFO] HTTP Request: GET https://huggingface.co/api/models/BarryKim34/kr-electra-political-bias/commits/refs%2Fpr%2F1 "HTTP/1.1 200 OK"
14:38:55 [INFO] HTTP Request: HEAD https://huggingface.co/BarryKim34/kr-electra-political-bias/resolve/refs%2Fpr%2F1/model.safetensors.index.json "HTTP/1.1 404 Not Found"
14:38:55 [INFO] HTTP Request: HEAD https://huggingface.co/BarryKim34/kr-electra-political-bias/resolve/refs%2Fpr%2F1/model.safetensors "HTTP/1.1 302 Found"


## 셀 4. 저장 헬퍼

In [4]:
from datetime import datetime
from email.utils import parsedate_to_datetime

def select_articles(articles: list, pro=3, con=3, neutral=2) -> list:
    """
    stance 기준으로 최신순 대표 기사 선별.
    pro/con/neutral 각각 최신순 정렬 후 지정 수만큼 추출.
    """
    def parse_date(s):
        try:
            return parsedate_to_datetime(str(s))
        except Exception:
            try:
                return datetime.fromisoformat(str(s))
            except Exception:
                return datetime.min

    sorted_arts = sorted(articles, key=lambda a: parse_date(a.get("published_at", "")), reverse=True)

    result  = [a for a in sorted_arts if a.get("stance") == "pro"][:pro]
    result += [a for a in sorted_arts if a.get("stance") == "con"][:con]
    result += [a for a in sorted_arts if a.get("stance") == "neutral"][:neutral]
    return result


def save_card(out_dir: Path, card: dict, keyword: str) -> Path:
    """
    카드 결과를 JSON으로 저장.
    파일명: {keyword}_{timestamp}.json
    """
    ts    = datetime.now().strftime("%Y%m%d_%H%M%S")
    safe  = keyword.replace("/", "_").replace(" ", "_")
    fpath = out_dir / f"{safe}_{ts}.json"
    with open(fpath, "w", encoding="utf-8") as f:
        json.dump(card, f, ensure_ascii=False, indent=2)
    return fpath


def build_policy_source(policy: dict) -> dict:
    """gov24 정책 딕셔너리 → FastAPI source 페이로드"""
    return {
        "id":      policy.get("서비스ID", ""),
        "name":    policy.get("서비스명", ""),
        "content": "\n".join(filter(None, [
            policy.get("서비스목적", ""),
            policy.get("지원내용", ""),
            policy.get("선정기준", ""),
        ])),
        "target":  policy.get("지원대상", policy.get("선정기준", ""))[:255],
        "method":  policy.get("신청방법", "")[:255],
        "period":  policy.get("신청기한", "")[:255],
        "contact": policy.get("문의처", "")[:255],
        "org":     policy.get("소관기관명", "")[:255],
        "url":     policy.get("온라인신청사이트URL", ""),
    }


def build_article_payload(articles: list) -> list:
    """기사 리스트 → FastAPI articles 페이로드"""
    return [
        {
            "title":     a.get("title", ""),
            "content":   a.get("content", ""),
            "url":       a.get("url", ""),
            "publisher": a.get("publisher", ""),
        }
        for a in articles
    ]

print("✅ 헬퍼 함수 준비 완료")

✅ 헬퍼 함수 준비 완료


## 셀 5. 뉴스 카드 단건 테스트

`KEYWORD` 값을 바꿔서 원하는 키워드 테스트

In [5]:
# ══════════════════════════════════════════════
# 테스트할 키워드 선택
print("사용 가능한 키워드:")
for kw in NEWS_BY_KEYWORD:
    print(f"  - {kw} ({len(NEWS_BY_KEYWORD[kw])}건)")

KEYWORD  = list(NEWS_BY_KEYWORD.keys())[1]  # 변경 가능
all_arts = NEWS_BY_KEYWORD[KEYWORD]
articles = select_articles(all_arts, pro=4, con=4, neutral=2)

print(f"\n{"="*50}")
print(f"🧪 뉴스 카드 단건 테스트: [{KEYWORD}]")
print(f"  전체 기사: {len(all_arts)}건 → 선별: {len(articles)}건")
stance_cnt = {"pro": 0, "con": 0, "neutral": 0}
for a in articles:
    s = a.get("stance", "neutral")
    stance_cnt[s] = stance_cnt.get(s, 0) + 1
print(f"  stance 분포: {stance_cnt}")
for i, a in enumerate(articles, 1):
    print(f"  [{i}] ({a.get("stance","?")}) {a.get("publisher","?")} — {a.get("title","")[:40]}")
print(f"{"="*50}\n")

news_result = news_gen.run(
    articles = build_article_payload(articles),
    save     = False,
)

if news_result:
    fpath = save_card(OUT_NEWS_DIR, news_result, KEYWORD)
    tabs  = news_result["tabs"]
    summ  = tabs.get("SUMMARY", {})
    print(f"\n{"="*50}")
    print(f"✅ 뉴스 카드 생성 완료")
    print(f"  저장 위치 : {fpath.name}")
    print(f"  intro     : {news_result.get("intro", tabs.get("intro", ""))}")
    print(f"  제목      : {summ.get("title", "")}")
    print(f"  카테고리  : {summ.get("category", "")}")
    print(f"  요약      : {summ.get("summary_points", [])}")
    print(f"  CORE      : {len(tabs.get("CORE",""))}자")
    print(f"  OPINION   : {len(tabs.get("OPINION",[]))}개 언론사")
    for op in tabs.get("OPINION", []):
        print(f"    - {op.get("media","?")} ({len(op.get("stance",""))}자)")
else:
    print("❌ 뉴스 카드 생성 실패")

22:35:22 [INFO] HTTP Request: GET https://huggingface.co/api/models/BarryKim34/kr-electra-political-bias/commits/main "HTTP/1.1 200 OK"


사용 가능한 키워드:
  - 결혼세액공제 (526건)
  - 고유가 피해지원금 (660건)
  - 국민배당금 (356건)
  - 레버리지ETF (756건)
  - 설탕 부담금 (534건)

🧪 뉴스 카드 단건 테스트: [고유가 피해지원금]
  전체 기사: 660건 → 선별: 10건
  stance 분포: {'pro': 4, 'con': 4, 'neutral': 2}
  [1] (pro) kukinews — ‘빚투 38조’ 주가 10%↓ 8조 증발할 수도...정부 F4 회동
  [2] (pro) idjnews — 당진시 고유가 피해지원금 2차 지급률 92.4%
  [3] (pro) enewstoday — NH농협카드, '고유가 피해지원금' 신청 315만건 돌파
  [4] (pro) metroseoul — 행안부, 내달 '전남광주통합특별시'출범…검찰청 폐지 및 중수청·공소...
  [5] (con) hankookilbo — [사설] 3% 넘어선 물가, 금리 인상 실기 말되 서민 챙겨야
  [6] (con) thescoop — 더 깊어진 고물가의 늪…'봄' 빼앗긴 전통시장의 눈물 [분석+]
  [7] (con) kado — 강원 고유가 피해지원금 이의신청 6100건…선정 기준 혼선
  [8] (con) mt — "고유가 피해지원금 왜 못 써"…술집서 행패 부린 50대 입건
  [9] (neutral) safetimes — 중동발 유가 불안에도 … 정부 "물가 2.7% 내외 전망"
  [10] (neutral) viva100 — 경남도, 소상공인시장 경기동향(BSI)지수 체감 상승폭 최근 1년 중 최



22:35:22 [INFO] HTTP Request: GET https://huggingface.co/api/models/BarryKim34/kr-electra-political-bias/discussions?p=0 "HTTP/1.1 200 OK"
22:35:22 [INFO] HTTP Request: GET https://huggingface.co/api/models/BarryKim34/kr-electra-political-bias/commits/refs%2Fpr%2F1 "HTTP/1.1 200 OK"
22:35:22 [INFO] HTTP Request: HEAD https://huggingface.co/BarryKim34/kr-electra-political-bias/resolve/refs%2Fpr%2F1/model.safetensors.index.json "HTTP/1.1 404 Not Found"
22:35:22 [INFO] HTTP Request: HEAD https://huggingface.co/BarryKim34/kr-electra-political-bias/resolve/refs%2Fpr%2F1/model.safetensors "HTTP/1.1 302 Found"
22:35:27 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
22:35:33 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
22:35:39 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
22:35:45 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
22:35

❌ 뉴스 카드 생성 실패


## 셀 6. 정책 카드 단건 테스트

`CAT_IDX`, `POL_IDX` 값을 바꿔서 테스트

In [5]:
# ══════════════════════════════════════════════
# 테스트할 정책 선택
print("카테고리 목록:")
cats = list(POLICY_DATA.keys())
for i, cat in enumerate(cats):
    print(f"  [{i}] {cat} — {len(POLICY_DATA[cat])}건")

CAT_IDX = 2   # 카테고리 인덱스 (변경 가능)
POL_IDX = 1   # 카테고리 내 정책 인덱스 (변경 가능)

cat     = cats[CAT_IDX]
policy  = POLICY_DATA[cat][POL_IDX]
source  = build_policy_source(policy)

# 정책명으로 관련 뉴스 찾기
# 관련 뉴스 — keyword_matched로 찾기
pol_name     = policy.get("서비스명", "")
service_id   = policy.get("서비스ID", "")
related_arts = RELATED_BY_KEYWORD.get(pol_name, [])[:10]

# 관련 법령 — 서비스ID로 찾기
related_laws = [
    law for law in LAW_DATA
    if any(p.get("서비스ID") == service_id for p in law.get("관련정책", []))
][:3]


print(f"\n{"="*50}")
print(f"🧪 정책 카드 단건 테스트")
print(f"  카테고리  : {cat}")
print(f"  정책명    : {pol_name}")
print(f"  관련 뉴스 : {len(related_arts)}건")
print(f"{"="*50}\n")

policy_result = policy_gen.run(
    source           = source,
    related_articles = related_arts,   # build_article_payload 불필요
    related_laws     = related_laws,   # 신규
    card_type        = "POLICY",
    save             = False,
)

if policy_result:
    fpath = save_card(OUT_POLICY_DIR, policy_result, pol_name)
    tabs  = policy_result["tabs"]
    summ  = tabs.get("SUMMARY", {})
    print(f"\n{"="*50}")
    print(f"✅ 정책 카드 생성 완료")
    print(f"  저장 위치 : {fpath.name}")
    print(f"  제목      : {summ.get("title", "")}")
    print(f"  카테고리  : {summ.get("category", "")}")
    print(f"  요약      : {summ.get("summary_points", [])}")
    print(f"  CORE      : {len(tabs.get("CORE",""))}자")
    print(f"  OPINION   : {len(tabs.get("OPINION",[]))}개 입장")
else:
    print("❌ 정책 카드 생성 실패")

카테고리 목록:
  [0] 일자리 — 5건
  [1] 교육 — 5건
  [2] 주거 — 5건
  [3] 금융 — 5건
  [4] 생활복지 — 5건
  [5] 문화 — 5건

🧪 정책 카드 단건 테스트
  카테고리  : 주거
  정책명    : 청년월세 지원
  관련 뉴스 : 10건



23:20:18 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:20:21 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:20:25 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:20:31 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:20:34 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:20:37 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:20:37 [INFO] [extract_facts] 정책: '청년월세 지원' | 기사: 5건
23:20:40 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:20:40 [INFO] [generate_summary] title='청년을 위한 월세 부담 완화'
23:20:40 [INFO] [pro_branch] 생성 시도 1/3
23:20:40 [INFO] [con_branch] 생성 시도 1/3
23:20:42 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:20:42 [INFO] [pro_branch] Generator 3


✅ 정책 카드 생성 완료
  저장 위치 : 청년월세_지원_20260607_232058.json
  제목      : 
  카테고리  : 주거
  요약      : ['청년 무주택자의 주거비 부담 완화', '월 최대 20만원, 최대 24개월 지원', '온라인과 오프라인 신청 모두 가능', '소득 조건 충족 시 지원 대상']
  CORE      : 2448자
  OPINION   : 2개 입장


## 셀 7. 결과 미리보기

`TARGET`을 `news_result` 또는 `policy_result`로 변경

In [ ]:
TARGET = news_result  # policy_result 로 변경 가능

if TARGET and TARGET.get("tabs"):
    print(f"카드 타입: {TARGET.get("card_type")} | 제목: {TARGET.get("title")}\n")
    for tab_name, content in TARGET["tabs"].items():
        print(f"{"="*50}")
        print(f"[{tab_name}]")
        if isinstance(content, str):
            print(content[:800])
            if len(content) > 800:
                print(f"  ... (총 {len(content)}자)")
        elif isinstance(content, (dict, list)):
            print(json.dumps(content, ensure_ascii=False, indent=2)[:800])
        print()

## 셀 8. 배치 실행 — 뉴스 카드

키워드별로 카드 1개씩 생성. 시간이 오래 걸릴 수 있어요.

In [ ]:
NEWS_LIMIT = len(NEWS_BY_KEYWORD)  # 줄이려면 숫자로 변경

print(f"📂 뉴스 키워드: {len(NEWS_BY_KEYWORD)}개 (상위 {NEWS_LIMIT}개 생성)")
news_batch = []

for keyword, all_arts in list(NEWS_BY_KEYWORD.items())[:NEWS_LIMIT]:
    articles = select_articles(all_arts, pro=4, con=4, neutral=2)
    print(f"\n  ▶ [{keyword}] 전체 {len(all_arts)}건 → 선별 {len(articles)}건")
    result = news_gen.run(
        articles = build_article_payload(articles),
        save     = True,
    )
    if result:
        fpath = save_card(OUT_NEWS_DIR, result, keyword)
        news_batch.append(result)
        print(f"    ✅ {result["title"]} → {fpath.name}")
        print(f"       OPINION: {len(result["tabs"].get("OPINION",[]))}개 언론사")
    else:
        print(f"    ❌ 생성 실패")

print(f"\n뉴스 카드 완료: {len(news_batch)}/{NEWS_LIMIT}개")
print(f"저장 위치: {OUT_NEWS_DIR}")

23:33:55 [INFO] HTTP Request: GET https://huggingface.co/api/models/BarryKim34/kr-electra-political-bias/commits/main "HTTP/1.1 200 OK"


📂 뉴스 키워드: 5개 (상위 5개 생성)

  ▶ [결혼세액공제] 전체 526건 → 선별 10건


23:33:56 [INFO] HTTP Request: GET https://huggingface.co/api/models/BarryKim34/kr-electra-political-bias/discussions?p=0 "HTTP/1.1 200 OK"
23:33:56 [INFO] HTTP Request: GET https://huggingface.co/api/models/BarryKim34/kr-electra-political-bias/commits/refs%2Fpr%2F1 "HTTP/1.1 200 OK"
23:33:56 [INFO] HTTP Request: HEAD https://huggingface.co/BarryKim34/kr-electra-political-bias/resolve/refs%2Fpr%2F1/model.safetensors.index.json "HTTP/1.1 404 Not Found"
23:33:56 [INFO] HTTP Request: HEAD https://huggingface.co/BarryKim34/kr-electra-political-bias/resolve/refs%2Fpr%2F1/model.safetensors "HTTP/1.1 302 Found"
23:34:00 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:34:06 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:34:13 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:34:20 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:34

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
23:35:31 [INFO] HTTP Request: HEAD https://huggingface.co/jhgan/ko-sroberta-multitask/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
23:35:31 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/jhgan/ko-sroberta-multitask/1050bd4e2ca90c0b9b62f0c1fbd83edc85ba8483/config.json "HTTP/1.1 200 OK"
23:35:31 [INFO] HTTP Request: HEAD https://huggingface.co/jhgan/ko-sroberta-multitask/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
23:35:31 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/jhgan/ko-sroberta-multitask/1050bd4e2ca90c0b9b62f0c1fbd83edc85ba8483/tokenizer_config.json "HTTP/1.1 200 OK"
23:35:31 [INFO] H

...

23:35:38 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:35:43 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:35:48 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:35:53 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:35:59 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:36:04 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:36:13 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:36:18 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:36:25 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:36:35 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:36:35 [INFO] [extract_facts] 10개 기사 사

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

23:37:31 [INFO] [save] 뉴스 카드 #99178366 Qdrant 저장 완료


    ✅ 신용거래융자 증가와 금융위기 경고 → 고유가_피해지원금_20260607_233731.json
       OPINION: 4개 언론사

  ▶ [국민배당금] 전체 356건 → 선별 10건


23:37:37 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:37:42 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:37:46 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:37:52 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:37:56 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:38:02 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:38:07 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:38:12 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:38:17 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:38:22 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
23:38:22 [INFO] [extract_facts] 10개 기사 사

## 셀 9. 배치 실행 — 정책 카드

In [ ]:
POLICY_LIMIT = total_policies  # 생성할 정책 수 (줄이려면 숫자로 변경)

print(f"📂 전체 정책: {total_policies}건 (상위 {POLICY_LIMIT}개 생성)")
policy_batch = []
count = 0

for cat, policies in POLICY_DATA.items():
    for policy in policies:
        if count >= POLICY_LIMIT:
            break
        pol_name     = policy.get("서비스명", "")
        source       = build_policy_source(policy)
        related_arts = RELATED_BY_KEYWORD.get(pol_name, [])[:10]

        print(f"\n  ▶ [{cat}] {pol_name} (관련뉴스 {len(related_arts)}건)...")
        result = policy_gen.run(
            source           = source,
            related_articles = build_article_payload(related_arts),
            card_type        = "POLICY",
            save             = True,
        )
        if result:
            fpath = save_card(OUT_POLICY_DIR, result, pol_name)
            policy_batch.append(result)
            print(f"    ✅ {result["title"]} → {fpath.name}")
        else:
            print(f"    ❌ 생성 실패")
        count += 1

print(f"\n정책 카드 완료: {len(policy_batch)}/{POLICY_LIMIT}개")
print(f"저장 위치: {OUT_POLICY_DIR}")

## 셀 10. 저장 파일 확인

In [6]:
print("=== data/cards/news/ ===")
for f in sorted(OUT_NEWS_DIR.glob("*.json")):
    data = json.loads(f.read_text(encoding="utf-8"))
    title = data.get("title", data.get("tabs", {}).get("SUMMARY", {}).get("title", "?"))
    print(f"  {f.name}  →  {title}")

print("\n=== data/cards/policy/ ===")
for f in sorted(OUT_POLICY_DIR.glob("*.json")):
    data = json.loads(f.read_text(encoding="utf-8"))
    title = data.get("title", data.get("tabs", {}).get("SUMMARY", {}).get("title", "?"))
    print(f"  {f.name}  →  {title}")

=== data/cards/news/ ===
  결혼세액공제_20260607_173224.json  →  세제 혜택으로 청년 고용 및 출산율 증진
  결혼세액공제_20260607_181425.json  →  청년 위한 결혼·출산 정책 변화
  결혼세액공제_20260607_182506.json  →  신혼부부 지원, 결혼과 출산 고민 해결
  결혼세액공제_20260607_183049.json  →  저출산 시대, 청년을 위한 새로운 제안
  결혼세액공제_20260607_183529.json  →  청년과 결혼, 웃음의 세금 공제
  결혼세액공제_20260607_184434.json  →  세제혜택으로 인구위기 극복?
  결혼세액공제_20260607_205933.json  →  저출산 문제와 세제 혜택
  결혼세액공제_20260607_212454.json  →  
  결혼세액공제_20260607_213231.json  →  혼인지속특별공제로 인구 위기 해결
  결혼세액공제_20260607_213551.json  →  인구위기 해결을 위한 세제 개혁
  결혼세액공제_20260607_215303.json  →  인구 감소 위기 속의 새로운 정책
  결혼세액공제_20260607_220455.json  →  인구위기 해법: 혁신투자와 포용정책
  국민배당금_20260607_213821.json  →  트럼프, AI 기업 지분 확보 추진!
  국민배당금_20260607_215553.json  →  트럼프, AI 기업 지분 논의
  국민배당금_20260607_220755.json  →  AI 수익, 국민에게 돌아올까?
  레버리지ETF_20260607_213943.json  →  개인투자자, 레버리지 ETF 매수 열풍
  레버리지ETF_20260607_215712.json  →  개인투자자들의 공격적 ETF 투자
  설탕_부담금_20260607_221033.json  →  설탕세 도입, 무엇이 문제일까?
  설탕_부담금_20260607_223248.json  →  설탕세 논